# 00 — Baseline: carry-forward TVT_input

Predict TVT in the evaluation zone as the last observed `TVT_input` value for that well. No ML. This anchors a Kaggle leaderboard floor and validates the submission pipeline.

Runs both locally (paths under `data/raw/`) and on Kaggle (paths under `/kaggle/input/...`). On Kaggle the submission lands at `/kaggle/working/submission.csv` (the filename Kaggle's grader requires).

In [ ]:
from __future__ import annotations
from pathlib import Path
import pandas as pd

KAGGLE_INPUT = Path('/kaggle/input/rogii-wellbore-geology-prediction')
if KAGGLE_INPUT.exists():
    RAW = KAGGLE_INPUT
    OUT_DIR = Path('/kaggle/working')
    OUT_NAME = 'submission.csv'        # required by Kaggle grader
    print('Environment: Kaggle')
else:
    repo = Path.cwd().resolve()
    while not (repo / 'pyproject.toml').exists():
        repo = repo.parent
    RAW = repo / 'data' / 'raw'
    OUT_DIR = repo / 'submissions'
    OUT_NAME = '00_carry_forward_submission.csv'
    print('Environment: local — repo at', repo)
OUT_DIR.mkdir(exist_ok=True)
print('RAW resolves to:', RAW.resolve())
print('Will write to: ', OUT_DIR / OUT_NAME)

In [ ]:
sample = pd.read_csv(RAW / 'sample_submission.csv')
sample[['well', 'row_index']] = sample['id'].str.rsplit('_', n=1, expand=True)
sample['row_index'] = sample['row_index'].astype(int)
wells = sample['well'].unique()
print(f'{len(wells)} wells, {len(sample):,} rows in submission')

In [ ]:
def predict_well(well: str) -> pd.DataFrame:
    h = pd.read_csv(RAW / 'test' / f'{well}__horizontal_well.csv')
    last_known = h['TVT_input'].dropna().iloc[-1] if h['TVT_input'].notna().any() else 0.0
    pred = h['TVT_input'].copy()
    pred[pred.isna()] = last_known
    return pd.DataFrame({
        'id': [f'{well}_{i}' for i in range(len(h))],
        'tvt': pred.values,
    })

preds = pd.concat([predict_well(w) for w in wells], ignore_index=True)
print(preds.head())
print(f'Total predictions: {len(preds):,}')

In [ ]:
submission = sample[['id']].merge(preds, on='id', how='left')
missing = submission['tvt'].isna().sum()
assert missing == 0, f'{missing} ids in sample_submission have no prediction'
out_path = OUT_DIR / OUT_NAME
submission.to_csv(out_path, index=False)
print('Wrote', out_path, '—', submission.shape)
print(submission['tvt'].describe())

## Next steps

1. On Kaggle, click "Save Version" → "Save & Run All (Commit)" — the notebook will execute end-to-end and `submission.csv` lands in the kernel's output.
2. Open the committed version, click "Submit to Competition".
3. Use the leaderboard score as the floor — every subsequent model must beat it.